# 04 — Sentiment Model
### Brand Sentiment Monitor

Fine-tunes `cardiffnlp/twitter-roberta-base-sentiment-latest` on a 3-class
sentiment dataset and saves to `models/sentiment/` for the attribution engine.

**EDA decisions implemented here:**

| Finding | Decision implemented |
|---------|----------------------|
| 1  | Neutral absent in S140 → supplement from GoEmotions `pure_neutral` |
| 2  | `max_length = 128` (covers 95% of tweets) |
| 9  | No hand-crafted features — RoBERTa learns them from raw tokens |
| 10 | `drop_duplicates()` before split to prevent data leakage |
| 11 | Temporal bias (2009) — final live validation deferred to notebook 10 |
| 14 | GoEmotions neutral is clean — use only `n_labels == 1` rows |
| 17 | Stratified 80 / 10 / 10 split |
| 18 | Exact hyperparams from EDA: epochs=3, batch=32, lr=2e-5, warmup=0.06 |
| 19 | Non-English heuristic filter before split |
| 21 | Contractions already expanded in nb02 — `not` tokenises as own subword |
| 22 | Error analysis checks negation pattern in neg→pos misclassifications |

**Outputs written by this notebook:**
```
models/sentiment/config.json               model config
models/sentiment/model.safetensors         fine-tuned weights
models/sentiment/tokenizer_config.json     tokenizer
models/sentiment/vocab.json                tokenizer vocab
models/sentiment/best_checkpoint/          auto-saved best epoch
data/kaggle/splits/sentiment_train.csv     train split
data/kaggle/splits/sentiment_val.csv       val split
data/kaggle/splits/sentiment_test.csv      test split
outputs/reports/sentiment_eval.json        full evaluation report
outputs/visualizations/04_training_curves.png
outputs/visualizations/04_confusion_matrix.png
outputs/visualizations/04_per_class_metrics.png
```

---
Run notebooks 00 → 03 first. **T4 GPU required.**

## 0. Setup

> **Before running:** Runtime → Change runtime type → **T4 GPU**

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys, json, re, ast, warnings, random, shutil
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DRIVE_ROOT  = "/content/drive/MyDrive/brand-sentiment-monitor"
KAGGLE_PROC = os.path.join(DRIVE_ROOT, "data/kaggle/processed")
KAGGLE_SPL  = os.path.join(DRIVE_ROOT, "data/kaggle/splits")
MODEL_OUT   = os.path.join(DRIVE_ROOT, "models/sentiment")
OUTPUTS_VIZ = os.path.join(DRIVE_ROOT, "outputs/visualizations")
OUTPUTS_RPT = os.path.join(DRIVE_ROOT, "outputs/reports")
CKPT_DIR    = os.path.join(MODEL_OUT, "best_checkpoint")

for d in [KAGGLE_SPL, MODEL_OUT, CKPT_DIR, OUTPUTS_VIZ, OUTPUTS_RPT]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, DRIVE_ROOT)
sys.path.insert(0, os.path.join(DRIVE_ROOT, "src"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Paths set ✅")


Mounted at /content/drive
Paths set ✅


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU)"
print(f"Device : {DEVICE}  ({GPU_NAME})")
print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    print("⚠️  No GPU — switch runtime to T4 GPU, training will otherwise take ~10x longer.")


Device : cuda  (Tesla T4)
PyTorch: 2.10.0+cu128


## 1. Load Config from EDA

Every hyperparameter is read from `outputs/reports/feature_insights.json`
produced by notebook 01. Nothing in this notebook is hardcoded.

In [3]:
with open(os.path.join(OUTPUTS_RPT, "feature_insights.json")) as f:
    fi = json.load(f)

with open(os.path.join(OUTPUTS_RPT, "eda_metadata.json")) as f:
    meta = json.load(f)

cfg = fi["training_config"]["sentiment"]

BASE_MODEL    = fi["recommended_model"]["sentiment"]
EPOCHS        = int(cfg["epochs"])
BATCH_SIZE    = int(cfg["batch_size"])
LR            = float(cfg["lr"])
MAX_LENGTH    = int(cfg["max_length"])
WARMUP_RATIO  = float(cfg["warmup_ratio"])
FILTER_NON_EN = bool(fi["preprocessing_flags"]["filter_non_english"])
MIN_TEXT_LEN  = int(fi["preprocessing_flags"]["min_text_length"])

LABEL2ID  = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL  = {0: "negative", 1: "neutral", 2: "positive"}
NUM_LABELS = 3

print("Config loaded from feature_insights.json ✅")
print(f"  base_model   : {BASE_MODEL}")
print(f"  epochs       : {EPOCHS}")
print(f"  batch_size   : {BATCH_SIZE}")
print(f"  lr           : {LR}")
print(f"  max_length   : {MAX_LENGTH}")
print(f"  warmup_ratio : {WARMUP_RATIO}")
print(f"  filter_non_en: {FILTER_NON_EN}")
print(f"  min_text_len : {MIN_TEXT_LEN}")


Config loaded from feature_insights.json ✅
  base_model   : cardiffnlp/twitter-roberta-base-sentiment-latest
  epochs       : 3
  batch_size   : 32
  lr           : 2e-05
  max_length   : 128
  warmup_ratio : 0.06
  filter_non_en: True
  min_text_len : 15


## 2. Load and Prepare Data

Steps in order:
1. Load `s140_clean.csv` — 1.6M binary (neg / pos) tweets from notebook 02
2. Non-English heuristic filter — Finding 19
3. Drop duplicates — Finding 10 (prevents data leakage)
4. Load `goemotions_clean.csv` — extract `pure_neutral` rows — Findings 1 + 14
5. Combine, balance classes, create `label_id` column

In [4]:
# ── Load Sentiment140 ─────────────────────────────────────────────────────────
s140 = pd.read_csv(os.path.join(KAGGLE_PROC, "s140_clean.csv"))

# Keep only text rows that are actual text (not NaN, not too short)
s140 = s140[s140["text"].notna()].copy()
s140["text"] = s140["text"].astype(str)
s140 = s140[s140["text"].str.len() >= MIN_TEXT_LEN].copy()

# s140_clean.csv has polarity (0/4) and label (negative/positive) columns
s140 = s140[s140["label"].isin(["negative", "positive"])].copy()

print(f"S140 loaded: {len(s140):,} rows")
print(s140["label"].value_counts().to_string())


S140 loaded: 1,528,791 rows
label
negative    769911
positive    758880


In [5]:
# ── Non-English heuristic filter (Finding 19) ─────────────────────────────────
# Full langdetect on 1.6M rows takes ~45 min on Colab.
# Heuristic: drop rows where >40% of characters are non-ASCII.
# This catches Spanish / French / Portuguese / Arabic tweets reliably.
# langdetect is still installed (nb00 batch 3) but used here only on edge cases.

def non_ascii_ratio(text):
    t = str(text)
    if len(t) == 0:
        return 1.0
    return sum(1 for c in t if ord(c) > 127) / len(t)

if FILTER_NON_EN:
    s140["_nar"] = s140["text"].apply(non_ascii_ratio)
    before       = len(s140)
    s140         = s140[s140["_nar"] < 0.40].copy()
    s140         = s140.drop(columns=["_nar"])
    print(f"Non-English heuristic filter: dropped {before - len(s140):,} rows")
    print(f"Remaining: {len(s140):,}")
else:
    print("Non-English filter skipped (FILTER_NON_EN=False)")


Non-English heuristic filter: dropped 915 rows
Remaining: 1,527,876


In [6]:
# ── Drop duplicates (Finding 10) ──────────────────────────────────────────────
before = len(s140)
s140   = s140.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Exact duplicates dropped: {before - len(s140):,}")
print(f"After dedup: {len(s140):,}")


Exact duplicates dropped: 24,382
After dedup: 1,503,494


In [7]:
# ── GoEmotions pure_neutral supplement (Findings 1, 14) ─────────────────────
ge_raw = pd.read_csv(os.path.join(KAGGLE_PROC, "goemotions_clean.csv"))

# label_ids column is saved as a string representation of a list
ge_raw["label_ids"] = ge_raw["label_ids"].apply(
    lambda v: ast.literal_eval(str(v)) if pd.notna(v) else []
)

# GoEmotions 28-class order — neutral is index 27
EMOTION_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral",
]
NEUTRAL_IDX = EMOTION_LABELS.index("neutral")   # 27

# pure_neutral = labeled ONLY as neutral (Finding 14: clean, not garbage-bin)
pure_neutral = ge_raw[
    ge_raw["label_ids"].apply(lambda ids: ids == [NEUTRAL_IDX])
].copy()
pure_neutral["label"] = "neutral"

print(f"GoEmotions total rows    : {len(ge_raw):,}")
print(f"Pure neutral rows (n=1)  : {len(pure_neutral):,}")
print(f"NEUTRAL_IDX confirmed    : {NEUTRAL_IDX}  ({EMOTION_LABELS[NEUTRAL_IDX]})")


GoEmotions total rows    : 52,294
Pure neutral rows (n=1)  : 15,263
NEUTRAL_IDX confirmed    : 27  (neutral)


In [8]:
# ── Combine and balance ───────────────────────────────────────────────────────
# Target: equal neg / neu / pos classes
# neg and pos come from S140 (balanced 800k/800k after dedup)
# neutral is capped at min(neg_count, available_pure_neutral)

neg_count = len(s140[s140["label"] == "negative"])
pos_count = len(s140[s140["label"] == "positive"])
neu_avail = len(pure_neutral)

print(f"Class sizes before balancing:")
print(f"  negative : {neg_count:,}")
print(f"  positive : {pos_count:,}")
print(f"  neutral  : {neu_avail:,} available from GoEmotions")

# Cap at smallest class so all three are equal
min_class   = min(neg_count, pos_count, neu_avail)
print(f"\nBalancing to {min_class:,} per class  →  {min_class * 3:,} total rows")

df = pd.concat([
    s140[s140["label"] == "negative"].sample(min_class, random_state=SEED)[["text", "label"]],
    s140[s140["label"] == "positive"].sample(min_class, random_state=SEED)[["text", "label"]],
    pure_neutral.sample(min_class, random_state=SEED)[["text", "label"]],
], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

df["label_id"] = df["label"].map(LABEL2ID)

print(f"\nFinal balanced dataset: {len(df):,} rows")
print(df["label"].value_counts().to_string())


Class sizes before balancing:
  negative : 759,789
  positive : 743,705
  neutral  : 15,263 available from GoEmotions

Balancing to 15,263 per class  →  45,789 total rows

Final balanced dataset: 45,789 rows
label
negative    15263
positive    15263
neutral     15263


## 3. Train / Val / Test Split

Stratified 80 / 10 / 10 split — Finding 17.
Stratification ensures each split has exactly 33.3% of each class.
Splits are saved so notebook 10 can reload the test set without recreating it.

In [9]:
# 80% train, 10% val, 10% test — stratified on label_id
train_df, temp_df = train_test_split(
    df, test_size=0.20,
    stratify=df["label_id"],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    stratify=temp_df["label_id"],
    random_state=SEED,
)

print(f"{'Split':<8} {'Rows':>8}   class distribution")
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    dist = split["label"].value_counts().to_dict()
    print(f"  {name:<6} {len(split):>8,}   {dist}")

# Save splits — notebook 10 reloads test_df to avoid reprocessing 1.6M rows
train_df.to_csv(os.path.join(KAGGLE_SPL, "sentiment_train.csv"), index=False)
val_df.to_csv(os.path.join(KAGGLE_SPL,   "sentiment_val.csv"),   index=False)
test_df.to_csv(os.path.join(KAGGLE_SPL,  "sentiment_test.csv"),  index=False)
print("\nSplits saved → data/kaggle/splits/")


Split        Rows   class distribution
  train    36,631   {'negative': 12211, 'neutral': 12210, 'positive': 12210}
  val       4,579   {'positive': 1527, 'neutral': 1526, 'negative': 1526}
  test      4,579   {'neutral': 1527, 'negative': 1526, 'positive': 1526}

Splits saved → data/kaggle/splits/


## 4. Tokeniser Check

Load the tokeniser and verify it handles the key patterns identified in EDA
before we tokenise 2M+ texts.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f"Tokenizer : {BASE_MODEL}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

# Spot-check key EDA patterns
spot_checks = [
    ("do not buy this",             "negation — Finding 21"),
    ("ABSOLUTELY TERRIBLE quality", "ALL CAPS — Finding 3"),
    (":enraged_face: shoes broke",  "emoji token — Finding 3"),
    ("Nike quality fell apart",     "typical brand complaint"),
    ("love love love these shoes",  "repeated positive"),
]
print()
for text, note in spot_checks:
    toks    = tokenizer.tokenize(text)
    n_toks  = len(toks)
    # Finding 21: 'not' must be its own subword after contraction expansion
    not_ok  = ""
    if "not" in text.lower():
        has_not = any(t.lstrip("Ġ") == "not" for t in toks)
        not_ok  = f"  ← 'not' own token: {has_not}"
    print(f"  {n_toks:>3} tokens | {note:<35} | {text[:45]}{not_ok}")


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Tokenizer : cardiffnlp/twitter-roberta-base-sentiment-latest
Vocab size: 50,265

    4 tokens | negation — Finding 21               | do not buy this  ← 'not' own token: True
    9 tokens | ALL CAPS — Finding 3                | ABSOLUTELY TERRIBLE quality
    8 tokens | emoji token — Finding 3             | :enraged_face: shoes broke
    5 tokens | typical brand complaint             | Nike quality fell apart
    5 tokens | repeated positive                   | love love love these shoes


## 5. PyTorch Dataset & DataLoader

In [11]:
class SentimentDataset(Dataset):
    """Tokenise once at init time — avoids repeated tokenisation per batch."""

    def __init__(self, texts, label_ids, tokenizer, max_length):
        # texts is a pandas Series or list of strings
        self.encodings = tokenizer(
            texts if isinstance(texts, list) else texts.tolist(),
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )
        self.labels = torch.tensor(
            label_ids if isinstance(label_ids, list) else label_ids.tolist(),
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : self.encodings["input_ids"][idx],
            "attention_mask" : self.encodings["attention_mask"][idx],
            "labels"         : self.labels[idx],
        }

print("SentimentDataset class defined ✅")


SentimentDataset class defined ✅


In [12]:
# Tokenise all splits — ~10-15 min on Colab for 2M+ texts
print("Tokenising train split...")
train_ds = SentimentDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LENGTH)
print("Tokenising val split...")
val_ds   = SentimentDataset(val_df["text"],   val_df["label_id"],   tokenizer, MAX_LENGTH)
print("Tokenising test split...")
test_ds  = SentimentDataset(test_df["text"],  test_df["label_id"],  tokenizer, MAX_LENGTH)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=False,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=2, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=2, pin_memory=True,
)

print(f"\nTrain batches : {len(train_loader):,}  ({len(train_ds):,} samples)")
print(f"Val batches   : {len(val_loader):,}  ({len(val_ds):,} samples)")
print(f"Test batches  : {len(test_loader):,}  ({len(test_ds):,} samples)")
print("DataLoaders ready ✅")


Tokenising train split...
Tokenising val split...
Tokenising test split...

Train batches : 1,145  (36,631 samples)
Val batches   : 72  (4,579 samples)
Test batches  : 72  (4,579 samples)
DataLoaders ready ✅


## 6. Model, Optimiser, Scheduler

Base model: `cardiffnlp/twitter-roberta-base-sentiment-latest`
Pre-trained on 58M tweets — domain match for Sentiment140.
`ignore_mismatched_sizes=True` replaces the existing 3-class head
with a fresh 3-class head labelled neg / neu / pos.

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model     : {BASE_MODEL}")
print(f"Device    : {DEVICE}")
print(f"Params    : {total_p:,} total  |  {trainable_p:,} trainable")


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model     : cardiffnlp/twitter-roberta-base-sentiment-latest
Device    : cuda
Params    : 124,647,939 total  |  124,647,939 trainable


In [14]:
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss()   # no class weights — dataset is balanced

print(f"Optimiser    : AdamW  lr={LR}  weight_decay=0.01")
print(f"Scheduler    : linear warmup → linear decay")
print(f"Loss         : CrossEntropyLoss  (no weights — classes are balanced)")
print(f"Total steps  : {total_steps:,}")
print(f"Warmup steps : {warmup_steps:,}  ({WARMUP_RATIO*100:.0f}% of training)")


Optimiser    : AdamW  lr=2e-05  weight_decay=0.01
Scheduler    : linear warmup → linear decay
Loss         : CrossEntropyLoss  (no weights — classes are balanced)
Total steps  : 3,435
Warmup steps : 206  (6% of training)


## 7. Training Loop

In [15]:
def run_epoch(model, loader, optimizer, scheduler, criterion, device, is_train):
    """Run one full pass over loader. Returns (loss, accuracy, macro_f1, preds, labels)."""
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_n    = 0
    all_preds  = []
    all_labels = []

    with torch.set_grad_enabled(is_train):
        for batch in tqdm(loader, desc="train" if is_train else "eval ", leave=False):
            input_ids  = batch["input_ids"].to(device)
            attn_mask  = batch["attention_mask"].to(device)
            labels_gpu = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            loss    = criterion(outputs.logits, labels_gpu)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

            preds = outputs.logits.argmax(dim=-1)
            total_loss += loss.item() * labels_gpu.size(0)
            total_n    += labels_gpu.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(batch["labels"].tolist())   # always from CPU batch

    avg_loss = total_loss / total_n
    accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / total_n
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, accuracy, macro_f1, all_preds, all_labels

print("run_epoch() defined ✅")


run_epoch() defined ✅


In [16]:
history = {
    "train_loss": [], "val_loss": [],
    "train_f1":   [], "val_f1":   [],
    "train_acc":  [], "val_acc":  [],
}
best_val_f1 = 0.0
best_epoch  = 0

print(f"Starting training — {EPOCHS} epoch(s)")
print(f"{'Ep':<4} {'TrLoss':>8} {'VaLoss':>8} {'TrF1':>7} {'VaF1':>7} {'VaAcc':>7}  Note")
print("─" * 60)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1, _,          _          = run_epoch(
        model, train_loader, optimizer, scheduler, criterion, DEVICE, is_train=True
    )
    va_loss, va_acc, va_f1, val_preds, val_labels   = run_epoch(
        model, val_loader,   optimizer, scheduler, criterion, DEVICE, is_train=False
    )

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_f1"].append(tr_f1)
    history["val_f1"].append(va_f1)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    is_best = va_f1 > best_val_f1
    if is_best:
        best_val_f1 = va_f1
        best_epoch  = epoch
        model.save_pretrained(CKPT_DIR)
        tokenizer.save_pretrained(CKPT_DIR)

    note = "✅ best saved" if is_best else ""
    print(f"  {epoch:<2}  {tr_loss:>8.4f} {va_loss:>8.4f} {tr_f1:>7.4f} {va_f1:>7.4f} {va_acc:>7.4f}  {note}")

print(f"\nBest val F1 = {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Best checkpoint → {CKPT_DIR}")


Starting training — 3 epoch(s)
Ep     TrLoss   VaLoss    TrF1    VaF1   VaAcc  Note
────────────────────────────────────────────────────────────


train:   0%|          | 0/1145 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

eval :   0%|          | 0/72 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  2     0.3191   0.4279  0.8775  0.8407  0.8408  ✅ best saved


train:   0%|          | 0/1145 [00:00<?, ?it/s]

eval :   0%|          | 0/72 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  3     0.2160   0.4878  0.9212  0.8432  0.8432  ✅ best saved

Best val F1 = 0.8432 at epoch 3
Best checkpoint → /content/drive/MyDrive/brand-sentiment-monitor/models/sentiment/best_checkpoint


## 8. Training Curves

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep_x = list(range(1, EPOCHS + 1))

# Loss
axes[0].plot(ep_x, history["train_loss"], "o-", label="Train", color="#3498db")
axes[0].plot(ep_x, history["val_loss"],   "o-", label="Val",   color="#e74c3c")
axes[0].set_title("Loss per Epoch", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)
for ep, (tl, vl) in enumerate(zip(history["train_loss"], history["val_loss"]), 1):
    axes[0].annotate(f"{tl:.3f}", (ep, tl), textcoords="offset points",
                     xytext=(0, 6), ha="center", fontsize=8, color="#3498db")
    axes[0].annotate(f"{vl:.3f}", (ep, vl), textcoords="offset points",
                     xytext=(0, -14), ha="center", fontsize=8, color="#e74c3c")

# F1
axes[1].plot(ep_x, history["train_f1"], "o-", label="Train F1", color="#3498db")
axes[1].plot(ep_x, history["val_f1"],   "o-", label="Val F1",   color="#e74c3c")
axes[1].plot(ep_x, history["val_acc"],  "s--",label="Val Acc",  color="#2ecc71", alpha=0.8)
axes[1].axhline(0.85, color="gray", ls=":", lw=1.5, label="Target ≥ 0.85")
axes[1].set_title("F1 & Accuracy per Epoch", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_ylim(0.0, 1.05)
axes[1].legend(); axes[1].grid(alpha=0.3)
for ep, vf in enumerate(history["val_f1"], 1):
    axes[1].annotate(f"{vf:.3f}", (ep, vf), textcoords="offset points",
                     xytext=(0, 6), ha="center", fontsize=8, color="#e74c3c")

plt.suptitle("Sentiment Model — Training History", fontweight="bold")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "04_training_curves.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

# Overfitting check
loss_gap = history["val_loss"][-1] - history["train_loss"][-1]
f1_gap   = history["train_f1"][-1] - history["val_f1"][-1]
print(f"Loss gap (val - train) last epoch: {loss_gap:+.4f}  "
      f"{'⚠️  possible overfit' if loss_gap > 0.15 else '✅ OK'}")
print(f"F1 gap  (trn - val ) last epoch: {f1_gap:+.4f}  "
      f"{'⚠️  possible overfit' if f1_gap > 0.10 else '✅ OK'}")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/04_training_curves.png
Loss gap (val - train) last epoch: +0.2719  ⚠️  possible overfit
F1 gap  (trn - val ) last epoch: +0.0780  ✅ OK


## 9. Test Set Evaluation

Load the best checkpoint (never seen during training).
Compute macro F1, per-class precision/recall/F1, and confusion matrix.

In [18]:
# Always reload from disk — guarantees we test the saved weights, not in-memory model
best_model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(DEVICE)
best_model.eval()

all_preds_t, all_labels_t, all_probs_t = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="test set"):
        input_ids  = batch["input_ids"].to(DEVICE)
        attn_mask  = batch["attention_mask"].to(DEVICE)
        labels_cpu = batch["labels"]            # keep on CPU for collection

        outputs = best_model(input_ids=input_ids, attention_mask=attn_mask)
        probs   = torch.softmax(outputs.logits, dim=-1)
        preds   = outputs.logits.argmax(dim=-1)

        all_preds_t.extend(preds.cpu().tolist())
        all_labels_t.extend(labels_cpu.tolist())
        all_probs_t.extend(probs.cpu().tolist())

TARGET_NAMES = ["negative", "neutral", "positive"]

report_str  = classification_report(
    all_labels_t, all_preds_t,
    target_names=TARGET_NAMES, digits=4,
)
report_dict = classification_report(
    all_labels_t, all_preds_t,
    target_names=TARGET_NAMES, output_dict=True, zero_division=0,
)

print("Test Set Classification Report:")
print(report_str)

macro_f1   = report_dict["macro avg"]["f1-score"]
target_met = macro_f1 >= 0.85
print(f"Macro F1 : {macro_f1:.4f}  "
      f"{'✅ Target met (≥ 0.85)' if target_met else '⚠️  Below target — review error analysis section'}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

test set:   0%|          | 0/72 [00:00<?, ?it/s]

Test Set Classification Report:
              precision    recall  f1-score   support

    negative     0.8197    0.8342    0.8269      1526
     neutral     0.8981    0.9181    0.9080      1527
    positive     0.8253    0.7923    0.8084      1526

    accuracy                         0.8482      4579
   macro avg     0.8477    0.8482    0.8478      4579
weighted avg     0.8477    0.8482    0.8478      4579

Macro F1 : 0.8478  ⚠️  Below target — review error analysis section


In [19]:
# Confusion matrix — raw counts + normalised
cm      = confusion_matrix(all_labels_t, all_preds_t)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in [
    (axes[0], cm,      "d",    "Counts"),
    (axes[1], cm_norm, ".2f",  "Row-normalised"),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES,
        ax=ax, linewidths=0.5,
    )
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {title}", fontweight="bold")

plt.suptitle("Sentiment Model — Test Set Confusion Matrix", fontweight="bold")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "04_confusion_matrix.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

# Key mistake numbers — used later in error analysis and saved to report
neg_as_pos = int(cm[0][2])   # true negative predicted as positive
neg_as_neu = int(cm[0][1])
pos_as_neg = int(cm[2][0])   # true positive predicted as negative
neu_acc_n  = int(cm[1][1])
print(f"\nKey misclassifications:")
print(f"  Negative → Positive : {neg_as_pos:,}  (critical: complaint looks positive)")
print(f"  Negative → Neutral  : {neg_as_neu:,}")
print(f"  Positive → Negative : {pos_as_neg:,}")
print(f"  Neutral correct     : {neu_acc_n:,} / {cm[1].sum():,}  ({neu_acc_n/cm[1].sum()*100:.1f}%)")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/04_confusion_matrix.png

Key misclassifications:
  Negative → Positive : 182  (critical: complaint looks positive)
  Negative → Neutral  : 71
  Positive → Negative : 229
  Neutral correct     : 1,402 / 1,527  (91.8%)


## 10. Error Analysis

EDA Finding 22 predicted that negated-positive phrases are the dominant
error mode for the negative class.
This section measures how much of the neg→pos error is caused by negation.

In [20]:
# Build error dataframe for all test misclassifications
test_texts_list = test_df["text"].reset_index(drop=True).tolist()

errors = []
for idx, (true_id, pred_id, probs) in enumerate(
    zip(all_labels_t, all_preds_t, all_probs_t)
):
    if true_id != pred_id:
        errors.append({
            "text"      : test_texts_list[idx],
            "true"      : ID2LABEL[true_id],
            "predicted" : ID2LABEL[pred_id],
            "confidence": round(max(probs), 4),
            "neg_prob"  : round(probs[0], 4),
            "neu_prob"  : round(probs[1], 4),
            "pos_prob"  : round(probs[2], 4),
        })

error_df = pd.DataFrame(errors)
total_test = len(all_labels_t)
print(f"Total test samples  : {total_test:,}")
print(f"Total errors        : {len(error_df):,}  ({len(error_df)/total_test*100:.2f}%)")
print()
print("Error counts by true → predicted:")
if len(error_df) > 0:
    print(error_df.groupby(["true", "predicted"]).size().to_string())


Total test samples  : 4,579
Total errors        : 695  (15.18%)

Error counts by true → predicted:
true      predicted
negative  neutral       71
          positive     182
neutral   negative      51
          positive      74
positive  negative     229
          neutral       88


In [21]:
# Negation pattern analysis on neg→pos errors (Finding 22)
NEGATION_PAT    = re.compile(
    r"\b(not|never|no\b|n't|hardly|barely|without|lacks?|missing)\b",
    re.IGNORECASE
)
NEGATED_POS_PAT = re.compile(
    r"\bnot\s+(good|great|happy|comfortable|amazing|love|nice|worth|recommend|satisfied|helpful)\b",
    re.IGNORECASE
)

# Initialise with safe defaults — used in section 12 report even if no errors
negation_pct    = 0.0
negated_pos_pct = 0.0
high_conf_errors_n = 0

if len(error_df) > 0:
    neg_pos_df = error_df[
        (error_df["true"] == "negative") & (error_df["predicted"] == "positive")
    ].copy()

    if len(neg_pos_df) > 0:
        neg_pos_df["has_negation"]    = neg_pos_df["text"].str.contains(NEGATION_PAT, regex=True)
        neg_pos_df["has_negated_pos"] = neg_pos_df["text"].str.contains(NEGATED_POS_PAT, regex=True)
        negation_pct    = neg_pos_df["has_negation"].mean() * 100
        negated_pos_pct = neg_pos_df["has_negated_pos"].mean() * 100

        print(f"Negative → Positive errors: {len(neg_pos_df):,}")
        print(f"  Has negation pattern    : {negation_pct:.1f}%")
        print(f"  Has negated-positive    : {negated_pos_pct:.1f}%  (not good / not happy etc)")
        print()
        if negation_pct > 20:
            print(f"✅ EDA Finding 22 confirmed — negation accounts for {negation_pct:.0f}% of neg→pos errors")
        else:
            print(f"ℹ️  Negation = {negation_pct:.0f}% of neg→pos errors — not dominant pattern")
            print("   RoBERTa may have learnt negation well. Check sarcasm coverage instead.")

        print()
        print("Highest-confidence neg→pos mistakes (most dangerous — complaint looks positive):")
        for _, row in neg_pos_df.nlargest(6, "confidence").iterrows():
            flag = " ← NEGATED" if row["has_negation"] else ""
            print(f"  conf={row['confidence']:.3f}  {row['text'][:85]}{flag}")
    else:
        print("✅ Zero negative→positive errors — model never mistakes complaints for praise")

    # High-confidence wrong predictions
    high_conf_errors = error_df[error_df["confidence"] > 0.90]
    high_conf_errors_n = len(high_conf_errors)
    print(f"\nHigh-confidence errors (> 0.90 confidence): {high_conf_errors_n}")
    if high_conf_errors_n > 0:
        print("  Sample:")
        for _, row in high_conf_errors.head(4).iterrows():
            print(f"    true={row['true']:<10} pred={row['predicted']:<10} conf={row['confidence']:.3f}")
            print(f"    {row['text'][:80]}")
else:
    print("✅ No errors on test set — perfect classification")
    neg_pos_df = pd.DataFrame()


Negative → Positive errors: 182
  Has negation pattern    : 19.2%
  Has negated-positive    : 0.0%  (not good / not happy etc)

ℹ️  Negation = 19% of neg→pos errors — not dominant pattern
   RoBERTa may have learnt negation well. Check sarcasm coverage instead.

Highest-confidence neg→pos mistakes (most dangerous — complaint looks positive):
  conf=0.997  Goodnight twitter
  conf=0.997  Hey Lizzie! Do you actually know when someone sends a message/replies to you on Twitt
  conf=0.996  Want to torture a lazy person? Tune TV to some crappy channel and keep the remote out
  conf=0.996  Wow, sounds awesome
  conf=0.996  Joey seems to know what I am talking about
  conf=0.995  haha I saved one for you

High-confidence errors (> 0.90 confidence): 315
  Sample:
    true=negative   pred=neutral    conf=0.999
    I think Hoosiers take some sort of dumb pride in pronouncing names of streets an
    true=negative   pred=neutral    conf=0.911
    Mine is so small I think the age thing is a bout rig

In [22]:
# Per-class metrics chart + error distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-class metrics
classes    = TARGET_NAMES
precs      = [report_dict[c]["precision"] for c in classes]
recs       = [report_dict[c]["recall"]    for c in classes]
f1s        = [report_dict[c]["f1-score"]  for c in classes]
x          = np.arange(len(classes))
w          = 0.25
bar_colors = ["#e74c3c", "#95a5a6", "#2ecc71"]

axes[0].bar(x - w, precs, w, label="Precision", color="#3498db", alpha=0.85)
axes[0].bar(x,     recs,  w, label="Recall",    color="#e67e22", alpha=0.85)
axes[0].bar(x + w, f1s,   w, label="F1",        color=bar_colors,alpha=0.85)
axes[0].axhline(0.85, color="black", ls="--", lw=1.5, label="Target 0.85")
axes[0].set_xticks(x)
axes[0].set_xticklabels(classes)
axes[0].set_ylim(0, 1.12)
axes[0].set_title("Per-Class Metrics — Test Set", fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)
for i, (p, r, f) in enumerate(zip(precs, recs, f1s)):
    axes[0].text(i - w, p + 0.01, f"{p:.2f}", ha="center", fontsize=8)
    axes[0].text(i,     r + 0.01, f"{r:.2f}", ha="center", fontsize=8)
    axes[0].text(i + w, f + 0.01, f"{f:.2f}", ha="center", fontsize=8)

# Error distribution
if len(error_df) > 0:
    err_grp    = error_df.groupby(["true", "predicted"]).size().reset_index(name="count")
    err_labels = [f"{r['true']}\n→{r['predicted']}" for _, r in err_grp.iterrows()]
    err_counts = err_grp["count"].tolist()
    axes[1].bar(err_labels, err_counts, color="#e74c3c", alpha=0.8, edgecolor="white", width=0.5)
    axes[1].set_title("Error Types — Test Set", fontweight="bold")
    axes[1].set_ylabel("Count")
    axes[1].grid(axis="y", alpha=0.3)
    for i, v in enumerate(err_counts):
        axes[1].text(i, v + max(err_counts) * 0.01, str(v), ha="center", fontsize=9)
else:
    axes[1].text(0.5, 0.5, "No errors!", ha="center", va="center",
                 transform=axes[1].transAxes, fontsize=16)
    axes[1].set_title("Error Types — Test Set", fontweight="bold")

plt.suptitle("Sentiment Model — Test Set Performance Summary", fontweight="bold")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "04_per_class_metrics.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/04_per_class_metrics.png


## 11. Save Final Model to `models/sentiment/`

Copy best checkpoint to the canonical path that `src/models/sentiment.py` reads from.
`SentimentModel.load()` will work after this cell.

In [ ]:
# Copy every file from CKPT_DIR to MODEL_OUT
# Remove any stale files from a previous run first
for fname in os.listdir(MODEL_OUT):
    fpath = os.path.join(MODEL_OUT, fname)
    if os.path.isfile(fpath):
        os.remove(fpath)

for fname in os.listdir(CKPT_DIR):
    shutil.copy2(
        os.path.join(CKPT_DIR, fname),
        os.path.join(MODEL_OUT, fname),
    )

# Inventory
saved = sorted(os.listdir(MODEL_OUT))
print(f"models/sentiment/ — {len(saved)} files:")
for fname in saved:
    sz = os.path.getsize(os.path.join(MODEL_OUT, fname)) / 1e6
    print(f"  {fname:<45}  {sz:>7.1f} MB")

required = {"config.json", "tokenizer_config.json"}
missing  = required - set(saved)
if missing:
    print(f"\n⚠️  Missing required files: {missing}")
else:
    print("\n✅ All required files present")


In [ ]:
# End-to-end reload + predict verification
# Uses the wrapper class from src/models/sentiment.py
from models.sentiment import SentimentModel

loaded_model = SentimentModel.load(MODEL_OUT)

verify_cases = [
    # (text,                                              expected_label)
    ("Nike just released the best shoe of the year",           "positive"),
    ("These shoes fell apart after one week, absolute garbage", "negative"),
    ("Nike reported quarterly earnings today",                  "neutral"),
    ("I do not recommend this product at all",                  "negative"),   # Finding 21
    ("WORST PURCHASE EVER completely useless",                  "negative"),   # ALL CAPS
    ("The quality is not what I expected, very disappointed",   "negative"),   # negation
]

print("End-to-end reload verification (SentimentModel.load → predict):")
print(f"  {'Expected':<12} {'Got':<12} {'Score':>7}  {'OK':>4}  Text")
print("  " + "─" * 80)
all_ok = True
for text, expected in verify_cases:
    result  = loaded_model.predict(text)
    correct = result["label"] == expected
    all_ok  = all_ok and correct
    status  = "✅" if correct else "❌"
    print(f"  {expected:<12} {result['label']:<12} {result['score']:>7.4f}  {status}  {text[:55]}")

print()
if all_ok:
    print("✅ All verify cases passed — SentimentModel.load() works correctly")
else:
    print("⚠️  Some unexpected predictions — review cases above")
    print("   Note: the model is probabilistic; borderline neutral cases may vary")


## 12. Save Evaluation Report

In [ ]:
# Build complete JSON report — notebook 10 reads this for live evaluation comparison
eval_report = {
    "model_path"  : "models/sentiment/",
    "base_model"  : BASE_MODEL,
    "num_labels"  : NUM_LABELS,
    "label_map"   : ID2LABEL,
    "training": {
        "epochs"       : EPOCHS,
        "best_epoch"   : best_epoch,
        "batch_size"   : BATCH_SIZE,
        "lr"           : LR,
        "max_length"   : MAX_LENGTH,
        "warmup_ratio" : WARMUP_RATIO,
        "n_train"      : len(train_df),
        "n_val"        : len(val_df),
        "n_test"       : len(test_df),
    },
    "val_best": {
        "macro_f1" : round(best_val_f1, 4),
        "epoch"    : best_epoch,
    },
    "test_results": {
        "macro_f1" : round(report_dict["macro avg"]["f1-score"], 4),
        "accuracy" : round(accuracy_score(all_labels_t, all_preds_t), 4),
        "per_class": {
            cls: {
                "precision": round(report_dict[cls]["precision"], 4),
                "recall"   : round(report_dict[cls]["recall"],    4),
                "f1"       : round(report_dict[cls]["f1-score"],  4),
                "support"  : int(report_dict[cls]["support"]),
            }
            for cls in TARGET_NAMES
        },
        "confusion_matrix": cm.tolist(),
    },
    "error_analysis": {
        "total_errors"               : len(error_df),
        "error_rate_pct"             : round(len(error_df) / total_test * 100, 3),
        "neg_predicted_as_pos"       : neg_as_pos,
        "pos_predicted_as_neg"       : pos_as_neg,
        "high_conf_errors_gt90"      : high_conf_errors_n,
        "negation_in_neg_pos_pct"    : round(negation_pct, 1),
        "negated_pos_in_neg_pos_pct" : round(negated_pos_pct, 1),
    },
    "target_met"   : macro_f1 >= 0.85,
    "training_history": history,
    "outputs": {
        "model_dir"             : "models/sentiment/",
        "training_curves_png"   : "outputs/visualizations/04_training_curves.png",
        "confusion_matrix_png"  : "outputs/visualizations/04_confusion_matrix.png",
        "per_class_metrics_png" : "outputs/visualizations/04_per_class_metrics.png",
        "eval_json"             : "outputs/reports/sentiment_eval.json",
        "train_split_csv"       : "data/kaggle/splits/sentiment_train.csv",
        "val_split_csv"         : "data/kaggle/splits/sentiment_val.csv",
        "test_split_csv"        : "data/kaggle/splits/sentiment_test.csv",
    },
}

rpt_path = os.path.join(OUTPUTS_RPT, "sentiment_eval.json")
with open(rpt_path, "w") as f:
    json.dump(eval_report, f, indent=2)

print(f"Saved → {rpt_path}")
print()
print("=" * 64)
print("SENTIMENT MODEL — FINAL SUMMARY")
print("=" * 64)
print(f"  Base model      : {BASE_MODEL}")
print(f"  Best epoch      : {best_epoch} / {EPOCHS}")
print(f"  Val macro F1    : {best_val_f1:.4f}")
print(f"  Test macro F1   : {macro_f1:.4f}  "
      f"{'✅ ≥ 0.85 target met' if target_met else '⚠️  < 0.85 — review error analysis'}")
for cls in TARGET_NAMES:
    f = report_dict[cls]['f1-score']
    p = report_dict[cls]['precision']
    r = report_dict[cls]['recall']
    print(f"  {cls:<10} F1={f:.4f}  P={p:.4f}  R={r:.4f}")
print(f"  Error rate      : {len(error_df)/total_test*100:.2f}%")
print(f"  neg→pos errors  : {neg_as_pos:,}  (complaints misclassified as positive)")
print(f"  Model saved     : models/sentiment/")
print("=" * 64)
print(f"\n✅ Notebook 04 complete. Next: 05_sarcasm_model.ipynb")
